# AI Resume Analyzer and Job Recommendation System

**Student:** Rida sahrin

**Project:** AI Resume Analyzer and Job Recommendation System (Unlox)

This notebook extracts a resume's content, identifies technical skills, compares the resume against a set of job roles using TF-IDF + cosine similarity, recommends the best-fit roles, and generates a skill-gap learning roadmap.

## Step 1: Install dependencies

In [ ]:
!pip install -q pypdf python-docx scikit-learn pandas

In [ ]:
import re
import pandas as pd
from pypdf import PdfReader
import docx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Step 2: Upload files
Upload THREE files when prompted: your resume (PDF or DOCX), `job_roles.csv`, and `skill_dictionary.csv` (both provided alongside this notebook).

In [17]:
from google.colab import files

uploaded = files.upload()
uploaded_names = list(uploaded.keys())
print("Uploaded:", uploaded_names)

resume_file = [f for f in uploaded_names if f.lower().endswith(('.pdf', '.docx'))][0]
print("Resume detected as:", resume_file)

Saving sample_resume.pdf to sample_resume (2).pdf
Uploaded: ['sample_resume (2).pdf']
Resume detected as: sample_resume (2).pdf


## Module 2: Text Extraction and Cleaning
Extract text from PDF or DOCX, lowercase it, remove extra symbols/spaces, but keep important technical symbols like C++, C#, .NET intact.

In [19]:
resume_file = "sample_resume.pdf"

In [20]:
def extract_text(filename):
    if filename.lower().endswith(".pdf"):
        reader = PdfReader(filename)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    elif filename.lower().endswith(".docx"):
        d = docx.Document(filename)
        return "\n".join(p.text for p in d.paragraphs)
    else:
        raise ValueError("Unsupported file type. Use PDF or DOCX.")

def clean_text(text):
    text = text.lower()
    # protect technical symbols before stripping punctuation
    text = text.replace("c++", "cplusplus").replace("c#", "csharp").replace(".net", "dotnet")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    # restore readable forms
    text = text.replace("cplusplus", "c++").replace("csharp", "c#").replace("dotnet", ".net")
    return text

raw_resume_text = extract_text(resume_file)
cleaned_resume_text = clean_text(raw_resume_text)

print("Extracted characters:", len(raw_resume_text))
print("\nCleaned text preview:\n", cleaned_resume_text[:400])

Extracted characters: 1185

Cleaned text preview:
 ananya rao ananya rao sample email com bengaluru india linkedin com in ananyarao sample education b sc computer science mount carmel college expected 2027 coursework data structures machine learning statistics database management systems technical skills programming python sql c++ data pandas numpy excel power bi machine learning scikit learn basic model building and evaluation tools git jupyter n


## Module 4: Load Job Role Dataset & Skill Dictionary

In [21]:
job_roles_df = pd.read_csv("job_roles.csv")
skill_dict_df = pd.read_csv("skill_dictionary.csv")

job_roles_df["required_skills"] = job_roles_df["required_skills"].apply(lambda s: [x.strip() for x in s.split(",")])
all_known_skills = sorted(skill_dict_df["skill"].str.lower().tolist())

print(f"Loaded {len(job_roles_df)} job roles and {len(all_known_skills)} known skills.")
job_roles_df.head()

Loaded 8 job roles and 44 known skills.


,job_role,required_skills
0,Data Analyst,"[python, sql, excel, pandas, power bi, data vi..."
1,Machine Learning Engineer,"[python, machine learning, scikit-learn, fasta..."
2,AI Engineer,"[python, deep learning, llm, rag, apis, pytorc..."
3,NLP Engineer,"[python, nlp, transformers, hugging face, spac..."
4,Computer Vision Engineer,"[python, opencv, cnn, yolo, pytorch, tensorflow]"


In [22]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=job_roles_df)

https://docs.google.com/spreadsheets/d/1QLu14dS3LZQDgSvgpXpj0DzfmzXTyueS_qNauizcr4Q/edit#gid=0


## Module 3: Skill Extraction
Search the cleaned resume text for every skill in the controlled skill dictionary, then group by category.

In [23]:
def extract_skills(text, known_skills):
    found = []
    for skill in known_skills:
        # word-boundary-safe search, works for multi-word skills too
        pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"
        if re.search(pattern, text):
            found.append(skill)
    return found

found_skills = extract_skills(cleaned_resume_text, all_known_skills)

skills_by_category = skill_dict_df[skill_dict_df["skill"].str.lower().isin(found_skills)]
print(f"Skills found: {len(found_skills)}\n")
for cat, group in skills_by_category.groupby("category"):
    print(f"{cat}: {', '.join(group['skill'].tolist())}")

Skills found: 14

data: pandas, numpy, excel, power bi, statistics
databases: sql
ml: machine learning, cnn
programming: python, c++
tools: git, streamlit, jupyter notebook, vs code


## Module 5: Matching and Recommendation
Convert the resume and each job role's required-skills text into TF-IDF vectors, then rank roles by cosine similarity.

In [24]:
resume_skill_text = " ".join(found_skills)
role_texts = [" ".join(skills) for skills in job_roles_df["required_skills"]]

corpus = [resume_skill_text] + role_texts
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

resume_vector = tfidf_matrix[0]
role_vectors = tfidf_matrix[1:]

similarities = cosine_similarity(resume_vector, role_vectors).flatten()

results_df = job_roles_df.copy()
results_df["match_score_%"] = (similarities * 100).round(1)
results_df = results_df.sort_values("match_score_%", ascending=False).reset_index(drop=True)

print("Ranked role matches:\n")
print(results_df[["job_role", "match_score_%"]])

Ranked role matches:

                    job_role  match_score_%
0               Data Analyst           45.6
1             Data Scientist           38.3
2  Machine Learning Engineer           26.9
3           Business Analyst           24.0
4           Python Developer           14.8
5   Computer Vision Engineer           13.2
6                AI Engineer            7.5
7               NLP Engineer            2.3


## Module 6: Skill-Gap Analysis & Roadmap
For a chosen target role, show which required skills were found vs. missing, and generate a simple week-by-week learning roadmap for the missing skills.

In [25]:
def skill_gap_analysis(target_role, results_df, found_skills):
    row = results_df[results_df["job_role"] == target_role].iloc[0]
    required = row["required_skills"]
    have = [s for s in required if s in found_skills]
    missing = [s for s in required if s not in found_skills]
    return have, missing

def generate_roadmap(missing_skills, weeks_per_skill=1):
    roadmap = []
    for i, skill in enumerate(missing_skills, start=1):
        roadmap.append(f"Week {i}: Learn the fundamentals of {skill} and build one small practice project.")
    return roadmap

# CHOOSE your target role from the ranked list above
target_role = results_df.iloc[0]["job_role"]  # defaults to the top match; change to any role name

have, missing = skill_gap_analysis(target_role, results_df, found_skills)
roadmap = generate_roadmap(missing)

print(f"Target Role: {target_role}")
print(f"Match Score: {results_df.iloc[0]['match_score_%']}%\n")
print("Skills Found:")
for s in have:
    print(" -", s)
print("\nMissing Skills:")
for s in missing:
    print(" -", s)
print("\nSuggested Roadmap:")
for line in roadmap:
    print(" ", line)

Target Role: Data Analyst
Match Score: 45.6%

Skills Found:
 - python
 - sql
 - excel
 - pandas
 - power bi
 - statistics

Missing Skills:
 - data visualization

Suggested Roadmap:
  Week 1: Learn the fundamentals of data visualization and build one small practice project.


## Full Report View
This mirrors the example output format from the project spec.

In [26]:
print("=" * 50)
print(f"RESUME ANALYSIS REPORT")
print("=" * 50)
print(f"\nTarget Role: {target_role}")
print(f"Match Score: {results_df.iloc[0]['match_score_%']}%\n")
print("Skills Found:")
for s in have:
    print(f"  - {s}")
print("\nMissing Skills:")
for s in missing:
    print(f"  - {s}")
print("\nTop 3 Recommended Roles:")
for i, row in results_df.head(3).iterrows():
    print(f"  {i+1}. {row['job_role']} - {row['match_score_%']}%")
print("\nSuggested Roadmap:")
for line in roadmap:
    print(f"  {line}")

RESUME ANALYSIS REPORT

Target Role: Data Analyst
Match Score: 45.6%

Skills Found:
  - python
  - sql
  - excel
  - pandas
  - power bi
  - statistics

Missing Skills:
  - data visualization

Top 3 Recommended Roles:
  1. Data Analyst - 45.6%
  2. Data Scientist - 38.3%
  3. Machine Learning Engineer - 26.9%

Suggested Roadmap:
  Week 1: Learn the fundamentals of data visualization and build one small practice project.


## Testing Sheet (for submission)

Test with at least 3 different resumes (or edit/re-upload variations) and record whether the recommended top role matches expectations.

| Test Resume | Expected Top Role | Actual Top Role | Correct? |
|---|---|---|---|
| Resume A |  |  |  |
| Resume B |  |  |  |
| Resume C |  |  |  |

## Responsible AI Notes
- This tool scores only job-related skills, education, projects, and experience -- it never reads or scores gender, age, religion, nationality, photos, marital status, or disability.
- Match scores are estimates to guide learning, not automatic hiring or rejection decisions.
- Missing keywords do not always mean missing ability -- a resume may simply phrase a skill differently.
- Uploaded resumes are processed only in this session and are not stored permanently.